# المختبر 1.1 · أداة الحل مقابل خطة المرسِل

شركة رَكْب للخدمات اللوجستية الحضرية (افتراضية). 120 طلبًا اصطناعيًا على طرق الرياض من OpenStreetMap، وثماني شاحنات صغيرة.

**لا تكتب كودًا.** شغّل كل خلية بالترتيب (Shift + Enter)، واقرأ ما تطبعه، وانقل الأرقام إلى `comparison_sheet.xlsx`. في الخلية 5 تغيّر مفاتيح من `False` إلى `True`، لا أكثر.

الأوقات بالدقائق من بداية الوردية (480 دقيقة). المسافات على الطرق الفعلية، والأزمنة بلا ازدحام.

## 1 · التجهيز
تثبّت الخلية أداة الحل (OR-Tools) إن لم تكن مثبتة، وتقرأ ملفات المختبر من المجلد نفسه إن وُجدت، وإلا نزّلتها من موقع المقرر.

In [ ]:
import os, sys, pathlib, subprocess, urllib.request

SITE = "https://sdaiaacademy.github.io/use-sec-205/labs/files/routing/"
FILES = ['rakb_routing.py', 'deliveries_120.csv', 'fleet.csv', 'distance_m.csv', 'duration_s.csv', 'manual_plan.csv', 'solver_plan_reference.csv']
try:
    import ortools  # noqa: F401
except ImportError:  # Colab and most fresh environments: install the solver once
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ortools==9.14.6206"], check=True)

here = pathlib.Path.cwd()
if not (here / "deliveries_120.csv").exists():
    here = pathlib.Path("rakb_routing_files")
    here.mkdir(exist_ok=True)
    try:
        for name in FILES:
            if not (here / name).exists():
                urllib.request.urlretrieve(SITE + name, here / name)
    except Exception as e:
        raise SystemExit("تعذّر تنزيل ملفات المختبر (" + str(e) + "). ارفع ملفات مجلد routing بجانب الدفتر ثم شغّل الخلية من جديد.")
sys.path.insert(0, str(here))
import rakb_routing as R
case = R.load(here)
FULL_FLEET = list(case["fleet"])
print("طلبات:", len(case["stops"]), "| شاحنات:", len(case["fleet"]), "| صناديق:", sum(s["crates"] for s in case["stops"]))

## 2 · خطة عايد اليدوية، بقواعده الأربع
نعيد «قيادة» الخطة: كل شاحنة تخرج في الدقيقة 0، وتنتظر إن وصلت قبل النافذة، وتأخذ راحتها، وتعود.

In [ ]:
manual = R.read_plan(here / "manual_plan.csv")
manual_rows, manual_sum = R.replay(case, manual)
manual_sum

## 3 · التشغيل الأول لأداة الحل
النموذج يعرف: السعات، ونوافذ التسليم، و5 دقائق لكل تسليم. ولا يعرف شيئًا آخر. التشغيل يأخذ ثواني.

In [ ]:
try:
    solver_plan = R.solve(case, rules=())
except Exception as e:  # إن تعذّر تشغيل أداة الحل، نبدأ من خطتها المحفوظة
    print("لم تعمل أداة الحل هنا:", e, "| نستخدم solver_plan_reference.csv")
    solver_plan = R.read_plan(here / "solver_plan_reference.csv")
print("كما تراها أداة الحل:", R.replay(case, solver_plan, rules=())[1])

## 4 · الخطة نفسها بقواعد عايد
الخطة لم تتغير. تغيّرت القواعد التي نحكم بها عليها. انقل الأرقام إلى العمود الثاني من جدول المقارنة، واقرأ قائمة المخالفات.

In [ ]:
solver_rows, solver_sum = R.replay(case, solver_plan)
print(solver_sum)
for r in solver_rows:
    if r["breach"] or r["late_min"]:
        print(r["van"], r["stop"], "| مخالفة:", r["breach"] or "-", "| تأخر بالدقائق:", r["late_min"])

## 5 · أعطِ أداة الحل ما يعرفه عايد
غيّر المفاتيح التي تراها مناسبة إلى `True`، ثم شغّل الخلية. القاعدة الرابعة (الراحة) لا تعرفها هذه النسخة من النموذج: ستراها في إعادة القيادة فقط.

In [ ]:
ADD_COLD = False      # البقالة المبرّدة على V8 فقط
ADD_LIFT = False      # الأثاث على V2 فقط
ADD_SERVICE = False   # 15 دقيقة في الأبراج
VANS = 8              # ✱ إضافي: جرّب 7

case["fleet"] = FULL_FLEET[:VANS]

rules = [name for name, on in (("cold", ADD_COLD), ("lift", ADD_LIFT), ("service", ADD_SERVICE)) if on]
plan2 = R.solve(case, rules=rules)
rows2, sum2 = R.replay(case, plan2)
print("القواعد في النموذج:", rules)
print(sum2)
for r in rows2:
    if r["breach"] or r["late_min"]:
        print(r["van"], r["stop"], "| مخالفة:", r["breach"] or "-", "| تأخر بالدقائق:", r["late_min"], "| راحة بعدها:", r["rest_after"] or "-")

## 6 · المقارنة في جدول واحد

In [ ]:
names = ["خطة عايد", "الأداة: التشغيل الأول", "الأداة: بعد القواعد"]
for key in ("km", "late_stops", "late_minutes", "vans_overtime", "cold_breaches", "lift_breaches"):
    print(f"{key:15}", *[f"{s[key]:>10}" for s in (manual_sum, solver_sum, sum2)])
R.write_plan("my_solver_plan.csv", plan2)
print("حُفظت خطتك في my_solver_plan.csv")

## ✱ إضافي
عُد إلى الخلية 5 واجعل `VANS = 7`، ثم شغّلها. هل تكفي سبع شاحنات؟ ما الذي ستقوله لمازن الشعيل عن الشاحنة الثامنة؟